In [5]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# -------------------------------------------------------------------------
# 1. DATA PREPARATION
# -------------------------------------------------------------------------
# Load dataset
df = pd.read_csv('netflix_revenue_updated.csv')
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df['Date_str'] = df['Date'].dt.strftime('%Y-%m')

# Extract key business metrics for executive summaries (KPI cards)
latest_revenue = df['Global Revenue'].iloc[-1] / 1e9  # In Billions

init_mem = df['Netflix Streaming Memberships'].iloc[0] / 1e6   # In Millions
final_mem = df['Netflix Streaming Memberships'].iloc[-1] / 1e6  # In Millions
mem_growth = ((final_mem - init_mem) / init_mem) * 100

# -------------------------------------------------------------------------
# 2. DASHBOARD ORCHESTRATION (PLOTLY)
# -------------------------------------------------------------------------
# Initialize a subplot grid: Rows 1 & 2 host individual interactive charts.
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.5, 0.5],
    vertical_spacing=0.12,
    subplot_titles=(
        "<b>Quarterly Global Revenue Trajectory</b>", 
        "<b>Global Streaming Membership Expansion</b>"
    )
)

# --- Chart 1: Global Revenue (Top Area/Line Chart) ---
fig.add_trace(
    go.Scatter(
        x=df['Date'],
        y=df['Global Revenue'],
        mode='lines+markers',
        name='Global Revenue',
        line=dict(color='#E50914', width=3),  # Brand color
        marker=dict(size=6, symbol='circle'),
        fill='tozeroy',
        fillcolor='rgba(229, 9, 20, 0.08)',   # Clean, transparent area accent
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Revenue:</b> $%{y:,.0f}<extra></extra>'
    ),
    row=1, col=1
)

# --- Chart 2: Membership Growth (Bottom Bar Chart) ---
fig.add_trace(
    go.Bar(
        x=df['Date'],
        y=df['Netflix Streaming Memberships'],
        name='Total Memberships',
        marker_color='#15803d',  # Balanced growth green
        opacity=0.85,
        hovertemplate='<b>Date:</b> %{x|%b %Y}<br><b>Memberships:</b> %{y:,.0f}<extra></extra>'
    ),
    row=2, col=1
)

# -------------------------------------------------------------------------
# 3. ENTERPRISE LAYOUT & KPI BANNER INJECTION
# -------------------------------------------------------------------------
fig.update_layout(
    title=dict(
        text="<b>Netflix Key Performance Indicators (Executive Dashboard)</b>",
        x=0.5,
        y=0.8,
        font=dict(size=20, color='#1e293b')
    ),
    showlegend=False,
    template='plotly_white',
    height=850,
    margin=dict(t=140, b=50, l=80, r=40),
    
    # Financial/Analytical KPI Highlights Banner positioned at the top
    annotations=[
        # Peak Revenue KPI Text
        dict(
            text=f"<b>LATEST PEAK REVENUE</b><br><span style='font-size:26px; color:#b91c1c;'>${latest_revenue:.2f}B</span><br><span style='color:#64748b; font-size:11px;'>As of {df['Date_str'].iloc[-1]}</span>",
            align='center', showarrow=False, xref='paper', yref='paper', x=0.20, y=1.13,
            bgcolor='#fef2f2', bordercolor='#fca5a5', borderwidth=1, borderpad=8
        ),
        # Membership Growth KPI Text
        dict(
            text=f"<b>MEMBERSHIP GROWTH</b><br><span style='font-size:26px; color:#15803d;'>+{mem_growth:.1f}%</span><br><span style='color:#64748b; font-size:11px;'>{init_mem:.1f}M → {final_mem:.1f}M subscribers</span>",
            align='center', showarrow=False, xref='paper', yref='paper', x=0.80, y=1.13,
            bgcolor='#f0fdf4', bordercolor='#bbf7d0', borderwidth=1, borderpad=8
        )
    ]
)

# Clean axis configurations optimized for time-series reading
fig.update_xaxes(showgrid=True, gridcolor='#e2e8f0', tickformat="%b %Y", row=1, col=1)
fig.update_yaxes(title_text="Revenue (USD)", showgrid=True, gridcolor='#e2e8f0', row=1, col=1)

fig.update_xaxes(showgrid=True, gridcolor='#e2e8f0', tickformat="%b %Y", row=2, col=1)
fig.update_yaxes(title_text="Subscribers", showgrid=True, gridcolor='#e2e8f0', row=2, col=1)

# Render Dashboard
fig.show()